# <span style="color:Thistle">Estructuras de datos en NumPy</span>

#### <span style="color:beige">Sesión 5</span>
---

Aunque a menudo nuestros datos pueden representarse bien mediante un array homogéneo de valores, a veces ese no es el caso. 

Vamos a ver ahora el uso de los *arrays estructurados* y los *arrays de registros* de NumPy, que proporcionan un <span style="color:Gold">almacenamiento eficiente</span> para datos compuestos y heterogéneos. 

Más adelante veremos también los `DataFrame` de Pandas.


In [1]:

import numpy as np

Imaginemos que tenemos varias categorías de datos sobre un conjunto de personas:

- nombre
- edad
- peso

Y queremos almacenar estos valores para usarlos en un script de Python. Hasta ahora los guardamos en tres arrays separados:

In [2]:
nombre = ['Alicia', 'Roberto', 'Catalina', 'Diego']
edad   = [25, 45, 37, 19]
peso   = [55.0, 85.5, 68.0, 61.5]

nombre, edad, peso

(['Alicia', 'Roberto', 'Catalina', 'Diego'],
 [25, 45, 37, 19],
 [55.0, 85.5, 68.0, 61.5])

Está bien, pero, no hay nada aquí que nos indique que los tres arrays están relacionados.

Sería más natural si pudiéramos usar <span style="color:Gold">una única estructura para almacenar todos estos datos</span>. NumPy puede manejar esto mediante `arrays estructurados`, que son arrays con tipos de datos compuestos.

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué es un array estructurado? Explicalo con tus propias palabras, no contestes arrays con tipos de datos compuestos.
</div>

En las sesiones anteriores, vimos como crear un array simple de esta manera:

In [3]:
x = np.zeros(4, dtype=int)

De manera similar, podemos crear un `array estructurado` usando `dtype` y la especificación del tipo de dato que lo compone:

In [4]:
# crear array estructurado
datos = np.zeros(4, dtype={'names':  ('nombre', 'edad', 'peso'),
                           'formats': ('U10',    'i4',   'f8')})
print(datos.dtype)

[('nombre', '<U10'), ('edad', '<i4'), ('peso', '<f8')]


- `'U10'`: Cadena Unicode de longitud máxima 10.
- `'i4'`: Entero de 4 bytes (es decir, 32 bits).
- `'f8'`: Flotante (_float_) de 8 bytes (es decir, 64 bits). 

Evidentemente, podriamos haber usado otras opciones para elegir los tipos de los datos.

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué contiene el array?
</div>

Ahora lo suyo es rellenarlo con nuestras listas de valores:

In [5]:
datos['nombre'] = nombre
datos['edad']   = edad
datos['peso']   = peso
print(datos)

[('Alicia', 25, 55. ) ('Roberto', 45, 85.5) ('Catalina', 37, 68. )
 ('Diego', 19, 61.5)]


Lo práctico de los arrays estructurados es que ahora puedes referirte a los valores tanto por índice como por nombre:

In [6]:
# Obtener todos los nombres
datos['nombre']

array(['Alicia', 'Roberto', 'Catalina', 'Diego'], dtype='<U10')

In [7]:
# Obtener la primera fila de datos
datos[0]

np.void(('Alicia', 25, 55.0), dtype=[('nombre', '<U10'), ('edad', '<i4'), ('peso', '<f8')])

In [10]:
# Obtener la última fila
datos[-1]

# Obtener el nombre de la última fila
datos[-1]['nombre']

np.str_('Diego')

<span style="color:Gold">Usando condiciones booleanas</span>, esto incluso permite realizar operaciones más sofisticadas, como filtrar por edad:

In [11]:
# Obtener nombres donde la edad sea menor de 30
datos[datos['edad'] < 30]['nombre']

array(['Alicia', 'Diego'], dtype='<U10')


---

## Creación de Arrays Estructurados

Los tipos de datos de los arrays estructurados pueden especificarse de varias maneras. Antes vimos el método con diccionario:

In [12]:
np.dtype({'names':  ('nombre', 'edad', 'peso'),
          'formats': ('U10',    'i4',   'f8')})

dtype([('nombre', '<U10'), ('edad', '<i4'), ('peso', '<f8')])

Para mayor claridad, los tipos numéricos pueden especificarse usando tipos de Python o `dtype`s de NumPy:

In [13]:
np.dtype({'names':  ('nombre', 'edad', 'peso'),
          'formats': ((np.str_, 10), int, np.float32)})

dtype([('nombre', '<U10'), ('edad', '<i8'), ('peso', '<f4')])

Un tipo compuesto también puede especificarse como una lista de tuplas:

In [15]:
np.dtype([('nombre', 'U10'), ('edad', 'i4'), ('peso', 'f8')])

dtype([('nombre', '<U10'), ('edad', '<i4'), ('peso', '<f8')])

Si los nombres de los tipos no te importan, puedes especificar solo los tipos en una cadena separada por comas:

In [16]:
np.dtype('S10,i4,f8')

dtype([('f0', 'S10'), ('f1', '<i4'), ('f2', '<f8')])

Los códigos de formato abreviados pueden parecer confusos, pero están construidos sobre principios sencillos. El primer carácter (opcional) es `<` o `>`, que significa "little endian" o "big endian" respectivamente, y especifica la convención de ordenación de los bits significativos. 

El siguiente carácter especifica el tipo de dato: caracteres, bytes, enteros, flotantes, etc. (véase la tabla a continuación). El último carácter o caracteres representa el tamaño del objeto en bytes.

| Carácter | Descripción | Ejemplo |
|---|---|---|
| `'b'` | Byte | `np.dtype('b')` |
| `'i'` | Entero con signo | `np.dtype('i4') == np.int32` |
| `'u'` | Entero sin signo | `np.dtype('u1') == np.uint8` |
| `'f'` | Punto flotante | `np.dtype('f8') == np.int64` |
| `'c'` | Punto flotante complejo | `np.dtype('c16') == np.complex128` |
| `'S'`, `'a'` | Cadena de bytes | `np.dtype('S5')` |
| `'U'` | Cadena Unicode | `np.dtype('U') == np.str_` |
| `'V'` | Datos en bruto (void) | `np.dtype('V') == np.void` |

---

## Tipos Compuestos Más Avanzados

Es posible definir tipos compuestos aún más avanzados. Por ejemplo, puedes crear un tipo donde cada elemento contiene un array o una matriz de valores. Aquí crearemos un tipo de dato con un campo `matriz` que consiste en una matriz flotante de $3 \times 3$:

In [17]:
tipo = np.dtype([('id', 'i8'), ('matriz', 'f8', (3, 3))])
X = np.zeros(1, dtype=tipo)
print(X[0])
print(X['matriz'][0])

(0, [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


Ahora cada elemento del array `X` consiste en:
- un `id`
- una `matriz` de $3 \times 3$. 

¿Por qué usarías esto en lugar de un simple array multidimensional o un diccionario de Python? Por lo mismo de siempre, la eficiencia.